# Field Description A/B Test

## Do Pydantic field descriptions actually change extraction quality?

This notebook proves a single claim: **field descriptions in Pydantic schemas are not just documentation -- they are instructions to the LLM.**

When you send a Pydantic schema to Claude for structured extraction, `model_json_schema()` serializes the entire schema -- including every `Field(description=...)` -- into the prompt. Claude reads those descriptions and uses them to decide what to put in each field.

The experiment is simple:

1. Define a **"weak" schema** with minimal field descriptions (just field names and types, no guidance).
2. Import the **"rich" schema** that the course provides (detailed descriptions on every field).
3. Run the **same extraction** on the **same documents** with both schemas.
4. Compare the results side by side.

If descriptions do not matter, the results should be identical. If they do matter, the rich schema should produce more precise, more consistent outputs -- especially on ambiguous documents.

**Think of it like form design.** A form with a box labeled "Name" gets wildly inconsistent entries. A form labeled "Full legal name (First Middle Last) as it appears on your government ID" gets exactly what you need. Pydantic schemas are the forms you hand to Claude — and Claude fills them out as literally as a distracted DMV applicant.

---

**Connection to AI-1:** This bridges prompt engineering (AI-1) to schema design (AI-2). In AI-1 you learned that how you ask the question shapes the answer. Here, the field descriptions *are* the question -- baked into your data model.

**How much does it matter?** In published benchmarks, changing a single field name in a schema swung model accuracy from 4.5% to 95%. Field design is not cosmetic — it is the primary quality lever for structured extraction.

**Time estimate:** 15-20 minutes to work through.

## 1. Setup

We need access to the `src` package (one directory up from `notebooks/`) and our API key from `.env`.

In [ ]:
import sys
import json
from pathlib import Path

# Add the project root so we can import from src
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel, Field
from src.s1_extraction.schemas import MemoExtraction
from src.s0_generation.generate import call_claude

print("Setup complete.")

## 2. Define the "Weak" Schema

This schema has the **same fields and types** as the rich schema, but with minimal descriptions. The field names alone tell Claude what data to extract -- nothing more.

Notice what is missing: no format guidance ("YYYY-MM-DD"), no clarification of what counts as an action item, no instruction to look for a priority level. Just bare field names.

In [ ]:
class WeakMemoExtraction(BaseModel):
    """Memo extraction with minimal field descriptions."""

    author: str
    date: str
    subject: str
    key_points: list[str]
    action_items: list[str] = []
    priority: str | None = None

print("WeakMemoExtraction defined.")

## 3. Compare the Schemas

Let us look at what Claude actually *sees* when we send each schema. The `model_json_schema()` method converts the Pydantic model into a JSON Schema -- this is what gets injected into the prompt.

Pay attention to the `description` fields (or lack thereof) in the weak version.

In [ ]:
weak_schema = WeakMemoExtraction.model_json_schema()
rich_schema = MemoExtraction.model_json_schema()

print("=" * 60)
print("WEAK SCHEMA (what Claude sees)")
print("=" * 60)
print(json.dumps(weak_schema, indent=2))
print()
print("=" * 60)
print("RICH SCHEMA (what Claude sees)")
print("=" * 60)
print(json.dumps(rich_schema, indent=2))

### What to notice

In the **weak** schema, properties look like:
```json
"date": { "title": "Date", "type": "string" }
```

In the **rich** schema, properties look like:
```json
"date": { "description": "Date in YYYY-MM-DD format", "title": "Date", "type": "string" }
```

That `description` text is the only difference -- and it is exactly the kind of prompt engineering you practiced in AI-1. The question is: does Claude use it?

## 4. Load Test Documents

We will use three Northbrook Partners memos that have different characteristics:

| Document | Why it is interesting |
|----------|----------------------|
| `memo_ceo_annual_kickoff.md` | Long memo with many potential "action items" that are really strategic goals -- tests whether the model distinguishes between aspirations and concrete follow-ups |
| `memo_security_update.md` | Clear action items with deadlines -- should be straightforward, but the date format and priority are implicit |
| `memo_office_relocation.md` | Mix of logistics and action items -- tests whether the model extracts the right subject and correctly identifies which items are action items vs. informational updates |

In [ ]:
DATA_DIR = Path("../data/northbrook")

docs = {
    "CEO Kickoff": (DATA_DIR / "memo_ceo_annual_kickoff.md").read_text(),
    "Security Update": (DATA_DIR / "memo_security_update.md").read_text(),
    "Office Relocation": (DATA_DIR / "memo_office_relocation.md").read_text(),
}

for name, text in docs.items():
    print(f"{name}: {len(text):,} characters, ~{len(text.split()):,} words")

## 4.5 Make Your Predictions

Before we run the experiment, commit to your predictions. This is not a quiz — it is a thinking exercise. You will learn more if you are wrong about something.

**For each question, write your answer in the cell below before running Section 5.**

In [ ]:
# YOUR PREDICTIONS — fill these in before running the experiment
# (No wrong answers here — this is about building intuition)

my_predictions = {
    # Which fields do you think will differ between weak and rich?
    # Choose from: author, date, subject, key_points, action_items, priority
    "fields_that_will_differ": ["date"],  # <-- edit this list

    # Which document will show the BIGGEST difference?
    # Choose: "CEO Kickoff", "Security Update", or "Office Relocation"
    "biggest_difference_doc": "CEO Kickoff",  # <-- edit this

    # Will the weak schema ever produce BETTER results than the rich one?
    "weak_ever_wins": False,  # <-- True or False
}

print("Your predictions recorded:")
for k, v in my_predictions.items():
    print(f"  {k}: {v}")
print("\nNow run Section 5 to see the results!")

## 5. The Experiment

We run the same extraction on each document twice: once with the weak schema, once with the rich schema.

We use `call_claude()` directly (rather than `extract_single()`) so we can control exactly which schema gets injected into the prompt. Both calls use the same system prompt, the same model, and the same temperature (0.0 for maximum consistency).

In [ ]:
import re

SYSTEM_PROMPT = (
    "You are a document extraction assistant. "
    "Return ONLY valid JSON matching the provided schema. "
    "No markdown, no explanation, just JSON."
)


def strip_markdown_json(text: str) -> str:
    """Remove markdown code fences if present."""
    stripped = text.strip()
    match = re.match(r"^```(?:json)?\s*\n?(.*?)\n?\s*```$", stripped, re.DOTALL)
    if match:
        return match.group(1).strip()
    return stripped


def extract_with_schema(document_text: str, schema_class: type[BaseModel]) -> dict:
    """Extract structured data from a document using a given schema.

    Returns the parsed dict (not a Pydantic object) so we can compare
    raw outputs without validation filtering the differences.
    """
    schema_json = json.dumps(schema_class.model_json_schema(), indent=2)

    user_prompt = (
        f"Extract data from the following document into this JSON schema:\n\n"
        f"--- SCHEMA ---\n{schema_json}\n\n"
        f"--- DOCUMENT ---\n{document_text}"
    )

    response_text = call_claude(
        prompt=user_prompt,
        system_prompt=SYSTEM_PROMPT,
        temperature=0.0,
    )

    cleaned = strip_markdown_json(response_text)
    return json.loads(cleaned)


print("Extraction function ready.")

In [ ]:
# Run the experiment -- this makes 6 API calls (2 schemas x 3 documents)
# Expected runtime: ~30-60 seconds

results = {}

for doc_name, doc_text in docs.items():
    print(f"\nExtracting: {doc_name}")

    print(f"  Running WEAK schema...")
    weak_result = extract_with_schema(doc_text, WeakMemoExtraction)

    print(f"  Running RICH schema...")
    rich_result = extract_with_schema(doc_text, MemoExtraction)

    results[doc_name] = {
        "weak": weak_result,
        "rich": rich_result,
    }

print("\nAll extractions complete.")

## 6. Compare Results

Now we compare the weak and rich extractions side by side for each document. We will look at each field individually so the differences are clear.

In [ ]:
def compare_results(doc_name: str, weak: dict, rich: dict):
    """Print a side-by-side comparison of weak vs rich extraction."""
    print("=" * 70)
    print(f"  {doc_name}")
    print("=" * 70)

    # Get all fields from both results
    all_fields = sorted(set(list(weak.keys()) + list(rich.keys())))

    for field in all_fields:
        weak_val = weak.get(field, "<missing>")
        rich_val = rich.get(field, "<missing>")

        # Check if values differ
        match = weak_val == rich_val
        marker = "  " if match else ">>"

        print(f"\n{marker} {field}:")

        if isinstance(weak_val, list) or isinstance(rich_val, list):
            # Print lists vertically for readability
            print(f"   WEAK:")
            if isinstance(weak_val, list):
                for item in weak_val:
                    print(f"     - {item}")
                if not weak_val:
                    print(f"     (empty list)")
            else:
                print(f"     {weak_val}")

            print(f"   RICH:")
            if isinstance(rich_val, list):
                for item in rich_val:
                    print(f"     - {item}")
                if not rich_val:
                    print(f"     (empty list)")
            else:
                print(f"     {rich_val}")
        else:
            print(f"   WEAK: {weak_val}")
            print(f"   RICH: {rich_val}")

    print()

### Document 1: CEO Annual Kickoff

This is a long, strategic memo. The interesting question: what counts as an "action item" vs. a "key point"? The memo mentions strategic goals, investment amounts, hiring plans, and an upcoming event. The rich schema's description says action items are "Follow-up tasks if any" -- will that help Claude distinguish between goals and tasks?

In [ ]:
compare_results(
    "CEO Annual Kickoff",
    results["CEO Kickoff"]["weak"],
    results["CEO Kickoff"]["rich"],
)

### The Hallucination Moment

Look at the `priority` field. The CEO Kickoff memo **never mentions a priority level**. But:

- **Weak schema** returned `"High"` — Claude *invented* a priority because nothing told it not to
- **Rich schema** returned `None` — the description says "Priority level **if mentioned**," so Claude correctly returned null

**This is a hallucination caused by schema design.** The weak schema gave Claude permission to guess. The rich schema told Claude to only extract what is explicitly stated. You already learned about hallucinations in AI-1 — now you see that your *data model* can cause or prevent them.

### Document 2: Security Update

This memo has very clear action items with explicit deadlines. The date field is interesting -- the memo is dated January 22, 2025. Will the weak schema return it in YYYY-MM-DD format, or in whatever format the document uses? The rich schema explicitly asks for "Date in YYYY-MM-DD format."

In [ ]:
compare_results(
    "Security Update",
    results["Security Update"]["weak"],
    results["Security Update"]["rich"],
)

### Document 3: Office Relocation

This memo mixes logistical updates with specific action items (pack your items, do not pack your computer, update your address). The weak schema gives no guidance on what "key points" vs. "action items" means -- will Claude lump everything together?

In [ ]:
compare_results(
    "Office Relocation",
    results["Office Relocation"]["weak"],
    results["Office Relocation"]["rich"],
)

## 6.5 The Date Test: Would Your Pipeline Survive?

The `date` field is the clearest proof point. Let's isolate it across all three documents.

In [ ]:
# Isolate the date field — the most unambiguous proof point

print("DATE FIELD COMPARISON")
print("=" * 60)
print(f"{'Document':<25}  {'Weak Schema':<25}  {'Rich Schema':<15}")
print("-" * 60)
for doc_name in results:
    weak_date = results[doc_name]["weak"].get("date", "?")
    rich_date = results[doc_name]["rich"].get("date", "?")
    match = "✓" if weak_date == rich_date else "✗"
    print(f"{doc_name:<25}  {weak_date:<25}  {rich_date:<15}  {match}")

print()
print("Now imagine parsing these in your pipeline:")
print()

# Show what happens when you try to parse weak dates
from datetime import datetime

print("Rich schema dates — consistent, parseable:")
for doc_name in results:
    rich_date = results[doc_name]["rich"].get("date", "?")
    try:
        parsed = datetime.strptime(rich_date, "%Y-%m-%d")
        print(f"  datetime.strptime('{rich_date}', '%Y-%m-%d')  →  {parsed.date()}  ✓")
    except ValueError as e:
        print(f"  datetime.strptime('{rich_date}', '%Y-%m-%d')  →  FAILED: {e}")

print()
print("Weak schema dates — inconsistent, fragile:")
for doc_name in results:
    weak_date = results[doc_name]["weak"].get("date", "?")
    try:
        parsed = datetime.strptime(weak_date, "%Y-%m-%d")
        print(f"  datetime.strptime('{weak_date}', '%Y-%m-%d')  →  {parsed.date()}  ✓")
    except ValueError as e:
        print(f"  datetime.strptime('{weak_date}', '%Y-%m-%d')  →  FAILED  ✗")

print()
print('Five words — "Date in YYYY-MM-DD format" — saved your pipeline from crashing.')

## 7. Summary: What Changed?

Let us count the differences across all three documents.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Build the comparison matrix
all_fields = ["author", "date", "subject", "key_points", "action_items", "priority"]
doc_names = list(results.keys())

# 1 = differ, 0 = match
matrix = []
for doc_name in doc_names:
    row = []
    for field in all_fields:
        weak_val = results[doc_name]["weak"].get(field, "<missing>")
        rich_val = results[doc_name]["rich"].get(field, "<missing>")
        row.append(1 if weak_val != rich_val else 0)
    matrix.append(row)

matrix = np.array(matrix)

# Create the heatmap
fig, ax = plt.subplots(figsize=(10, 4))

# Color: green for match, red for differ
from matplotlib.colors import ListedColormap
cmap = ListedColormap(["#22c55e", "#ef4444"])

im = ax.imshow(matrix, cmap=cmap, aspect="auto", vmin=0, vmax=1)

# Labels
ax.set_xticks(range(len(all_fields)))
ax.set_xticklabels(all_fields, fontsize=11, fontweight="bold")
ax.set_yticks(range(len(doc_names)))
ax.set_yticklabels(doc_names, fontsize=11)

# Add text annotations
for i in range(len(doc_names)):
    for j in range(len(all_fields)):
        label = "DIFFER" if matrix[i, j] else "MATCH"
        color = "white" if matrix[i, j] else "black"
        ax.text(j, i, label, ha="center", va="center",
                fontsize=9, fontweight="bold", color=color)

ax.set_title("Weak vs Rich Schema: Field-by-Field Scorecard",
             fontsize=14, fontweight="bold", pad=15)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#22c55e", label="Match"),
    Patch(facecolor="#ef4444", label="Differ"),
]
ax.legend(handles=legend_elements, loc="upper right",
          bbox_to_anchor=(1.15, 1.0), fontsize=10)

plt.tight_layout()
plt.show()

# Summary stats
total_fields = matrix.size
total_diffs = int(matrix.sum())
print(f"\nScorecard: {total_diffs}/{total_fields} fields differ across {len(doc_names)} documents.")
print(f"That is {total_diffs/total_fields:.0%} of all fields changed by adding descriptions.")

if total_diffs > 0:
    # Which fields always differ?
    always_differ = [f for j, f in enumerate(all_fields) if all(matrix[:, j])]
    if always_differ:
        print(f"\nFields that differ in EVERY document: {', '.join(always_differ)}")
    
    never_differ = [f for j, f in enumerate(all_fields) if not any(matrix[:, j])]
    if never_differ:
        print(f"Fields that NEVER differ: {', '.join(never_differ)}")

## 8. Common Differences to Look For

Even if some fields happen to match, here are the patterns that typically emerge when you compare weak vs. rich schemas:

| Field | Weak Schema Behavior | Rich Schema Behavior |
|-------|---------------------|---------------------|
| `date` | May return "January 6, 2025" or "Jan 6, 2025" -- whatever the document says | Returns "2025-01-06" -- the description says "YYYY-MM-DD format" |
| `key_points` | May include everything -- action items, goals, facts, stats | More focused on "important points from the memo" as the description instructs |
| `action_items` | May be empty, or may duplicate key_points | Better at identifying actual "follow-up tasks" as described |
| `priority` | May guess or return null inconsistently | Returns priority "if mentioned" -- null when the memo does not state one explicitly |
| `author` | May return just first name or include title | The description says "Who wrote the memo" -- typically returns full name |

The date field is usually the most visually obvious difference. But the `key_points` vs. `action_items` distinction is the most *meaningful* one for downstream use.

## 9. The Evidence, Summarized

One table combining what each description said and how many documents it affected:

In [ ]:
# Recap table: description text + impact

all_fields = ["author", "date", "subject", "key_points", "action_items", "priority"]
doc_names = list(results.keys())

print(f"{'Field':<15}  {'Rich Description':<45}  {'Docs Affected':>13}")
print("-" * 78)

for field in all_fields:
    # Get the rich description
    desc = rich_schema.get("properties", {}).get(field, {}).get("description", "(none)")
    
    # Count how many docs differ on this field
    diffs = sum(
        1 for doc in doc_names
        if results[doc]["weak"].get(field) != results[doc]["rich"].get(field)
    )
    
    marker = f"{diffs}/{len(doc_names)}"
    impact = " ◀ ALL" if diffs == len(doc_names) else ""
    print(f"{field:<15}  {desc:<45}  {marker:>13}{impact}")

print()
print("Each description is 3-8 words. Each one changed extraction behavior")
print("across multiple documents. This is the return on investment.")

## 10. The Takeaway

**Field descriptions are prompt engineering at the schema level.**

When you define a Pydantic model for LLM extraction, every `Field(description=...)` is an instruction to the model. The description tells Claude:

- **What format** to use ("YYYY-MM-DD" vs. free-form)
- **What counts** as a valid value ("follow-up tasks" vs. anything)
- **When to return null** ("if mentioned" vs. guessing)

This bridges two worlds:

- In **AI-1**, you learned that prompt phrasing matters -- the same question asked differently gets different answers.
- In **AI-2**, the prompt lives inside your schema. Good schema design *is* good prompt engineering.

### The practical rule

For every field in an extraction schema, write the description as if you are explaining to a careful human assistant what to put in that field. Be specific about:

1. **Format** -- dates, units, casing
2. **Scope** -- what belongs in this field vs. another field
3. **Edge cases** -- what to do when the value is missing or ambiguous

This costs you 10 seconds per field. It saves you hours of debugging bad extractions downstream.

## 11. Your Turn: Fix One Field [YELLOW Exercise]

Pick one field from the weak schema. Write a description for it. Re-run extraction on one document. Did the output change?

This takes 2-3 minutes. You are not building a full schema — just proving to yourself that you can steer extraction with a single sentence.

In [ ]:
# YELLOW exercise: improve one field in the weak schema

class ImprovedMemoExtraction(BaseModel):
    """Your improved schema — add a description to ONE field."""

    author: str
    date: str  # <-- Try: Field(description="Date in YYYY-MM-DD format")
    subject: str
    key_points: list[str]
    action_items: list[str] = []
    priority: str | None = None

# Pick one document to test with
test_doc_name = "CEO Kickoff"
test_doc_text = docs[test_doc_name]

# Run extraction with your improved schema
print(f"Extracting '{test_doc_name}' with your improved schema...\n")
improved_result = extract_with_schema(test_doc_text, ImprovedMemoExtraction)

# Compare all three versions
print(f"{'Field':<15}  {'Weak':<30}  {'Yours':<30}  {'Rich':<30}")
print("-" * 108)
for field in ["author", "date", "subject", "key_points", "action_items", "priority"]:
    weak_val = results[test_doc_name]["weak"].get(field, "?")
    your_val = improved_result.get(field, "?")
    rich_val = results[test_doc_name]["rich"].get(field, "?")
    
    # Truncate long values
    def fmt(v, width=28):
        s = str(v)
        return s[:width] + ".." if len(s) > width else s
    
    # Mark if yours matches rich (the goal)
    match = " ✓" if your_val == rich_val else ""
    print(f"{field:<15}  {fmt(weak_val):<30}  {fmt(your_val):<30}  {fmt(rich_val):<30}{match}")

print()
print("Did your description change the output?")
print("If not, try a more specific description. If yes — you just steered an LLM with 5 words.")